<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 Generator Audiovisual with SGLang

This notebook calls already-running SGLang Cosmos3 servers with direct `curl` requests from Python.

The examples are split into Cosmos3-Nano and Cosmos3-Super sections. Each section is self-contained, so you can run just one. Each section targets the matching model endpoint.


## 1. Prerequisites

Use a running SGLang server and set endpoint environment variables before the setup cell if you are not using the local default. Text-to-image uses `/v1/images/generations`; video modes use `/v1/videos`.

```bash
export COSMOS3_SGLANG_BASE_URL=http://localhost:30000
export COSMOS3_SGLANG_NANO_BASE_URL=http://localhost:30000
export COSMOS3_SGLANG_SUPER_BASE_URL=http://localhost:30000
```


## 2. Start the Server

Run the SGLang server before running the request cells. Use the Docker image for every modality on this page. Mount any directory that contains local media or action files you want the server to read.

### Docker Image: Cosmos3-Nano

```bash
docker run --runtime nvidia --gpus all \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  -v "$(pwd):/workspace" \
  -p 30000:30000 \
  --ipc=host \
  lmsysorg/sglang:dev \
  sglang serve \
  --model-path nvidia/Cosmos3-Nano \
  --host 0.0.0.0
```

### Docker Image: Cosmos3-Super

`Cosmos3-Super` is the larger 64B model, so it usually needs more GPU memory than `Cosmos3-Nano`. `--tp-size` splits model weights across multiple GPUs and reduces per-GPU memory use. Use `--performance-mode memory` preset to preserve memory.

For example, on four GPUs:

```bash
docker run --runtime nvidia --gpus all \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  -v "$(pwd):/workspace" \
  -p 30000:30000 \
  --ipc=host \
  lmsysorg/sglang:dev \
  sglang serve \
  --model-path nvidia/Cosmos3-Super \
  --host 0.0.0.0 \
  --num-gpus 4 \
  --performance-mode memory
```

### CFG Parallel

Use `--cfg-parallel-size 2` to run the positive and negative CFG branches in parallel on two GPUs:

```bash
sglang serve \
  --model-path nvidia/Cosmos3-Nano \
  --host 0.0.0.0 \
  --num-gpus 2 \
  --enable-cfg-parallel \
  --cfg-parallel-size 2
```

For Cosmos3, set CFG strength with the request-level `guidance_scale` field. Do not use `true_cfg_scale` for CFG Parallel with these Cosmos3 examples.


## 3. Configure Paths and Endpoints

This setup cell only configures repo/output paths and SGLang endpoint settings.


In [ ]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_AUDIOVISUAL_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "audiovisual"
COSMOS3_AUDIOVISUAL_OUTPUT_ROOT = Path(
    os.environ.get("COSMOS3_AUDIOVISUAL_OUTPUT_ROOT", COSMOS3_AUDIOVISUAL_ROOT / "outputs" / "notebooks")
).resolve()
DEFAULT_SGLANG_BASE_URL = os.environ.get("COSMOS3_SGLANG_BASE_URL", "http://localhost:30000")
SGLANG_ENDPOINTS = {
    "Cosmos3-Nano": os.environ.get("COSMOS3_SGLANG_NANO_BASE_URL", DEFAULT_SGLANG_BASE_URL),
    "Cosmos3-Super": os.environ.get("COSMOS3_SGLANG_SUPER_BASE_URL", DEFAULT_SGLANG_BASE_URL),
}

os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"] = str(COSMOS3_AUDIOVISUAL_OUTPUT_ROOT)
os.environ.setdefault("COSMOS3_SGLANG_API_KEY", "")

print("COSMOS_ROOT:", COSMOS_ROOT)
print("COSMOS3_AUDIOVISUAL_OUTPUT_ROOT:", COSMOS3_AUDIOVISUAL_OUTPUT_ROOT)
for model, endpoint in SGLANG_ENDPOINTS.items():
    print(f"{model} endpoint: {endpoint}")


## 4. Verify Endpoint Configuration


In [ ]:
from urllib.parse import urlparse

for model, base_url in SGLANG_ENDPOINTS.items():
    api_root = base_url.rstrip("/")
    if not api_root.endswith("/v1"):
        api_root = f"{api_root}/v1"
    parsed = urlparse(api_root)
    print(model)
    print("  api root:", api_root)
    print("  images generations:", f"{api_root}/images/generations")
    print("  videos async:", f"{api_root}/videos")
    print("  scheme:", parsed.scheme)
    print("  host:", parsed.netloc)


## 5. Preview Available Inputs


In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display

assets_dir = COSMOS3_AUDIOVISUAL_ROOT / "assets"
for prompt_dir in sorted((assets_dir / "prompts").iterdir()):
    if not prompt_dir.is_dir():
        continue
    print(f"{prompt_dir.relative_to(assets_dir)}:")
    for prompt_path in sorted(prompt_dir.glob("*.json")):
        data = json.loads(prompt_path.read_text())
        caption = (
            data.get("temporal_caption")
            or data.get("comprehensive_t2i_caption")
            or data.get("extra", {}).get("prompt", "")
        )
        print(f"  {prompt_path.name}: {caption[:180]}{'...' if len(caption) > 180 else ''}")
    print()

for image_dir in sorted((assets_dir / "images").iterdir()):
    if not image_dir.is_dir():
        continue
    print(f"{image_dir.relative_to(assets_dir)}:")
    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
            print(f"  {image_path.name}")
            display(Image(filename=str(image_path), width=420))


## 6. Define Asset Sets, Payload Helpers, Request Helpers, and Viewer Helpers


In [ ]:
import json
import os
from pathlib import Path
from IPython.display import Image, display, Video

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

FIXED_SAMPLING = {
    "num_steps": 35,
    "guidance": 6.0,
    "shift": 10.0,
    "fps": 24,
    "num_frames": 189,
    "resolution": "720",
    "aspect_ratio": "16,9",
    "seed": 0,
}

# All asset paths are repo-relative under cookbooks/cosmos3/generator/audiovisual.
# Model and sound choices live in this manifest; folders are organized only by modality.
ASSET_SETS = {
    "t2i": {
        "model": "Cosmos3-Nano",
        "mode": "text2image",
        "prompt": "assets/prompts/text2image/robot_draping.json",
        "enable_sound": False,
    },
    "t2i_super": {
        "model": "Cosmos3-Super",
        "mode": "text2image",
        "prompt": "assets/prompts/text2image/robot_draping.json",
        "enable_sound": False,
    },
    "t2v_nano_noaudio": {
        "model": "Cosmos3-Nano",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_kitchen.json",
        "enable_sound": False,
    },
    "t2vs": {
        "model": "Cosmos3-Nano",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_pouring_water_audio.json",
        "enable_sound": True,
    },
    "i2v_nano_noaudio": {
        "model": "Cosmos3-Nano",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/car_driving.json",
        "image": "assets/images/image2video/car_driving.jpg",
        "enable_sound": False,
    },
    "i2vs": {
        "model": "Cosmos3-Nano",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/coastal_road_audio.json",
        "image": "assets/images/image2video/coastal_road_audio.jpg",
        "enable_sound": True,
    },
    "v2vs": {
        "model": "Cosmos3-Nano",
        "mode": "video2video",
        "prompt": "assets/prompts/image2video/car_driving.json",
        "negative_prompt": "assets/negative_prompts/image2video/neg_prompt.json",
        "video": "assets/videos/car_driving_plain.mp4",
        "enable_sound": True,
    },
    "t2v_super_noaudio": {
        "model": "Cosmos3-Super",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_kitchen.json",
        "enable_sound": False,
    },
    "i2v_super_noaudio": {
        "model": "Cosmos3-Super",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/car_driving.json",
        "image": "assets/images/image2video/car_driving.jpg",
        "enable_sound": False,
    },
}


def asset_path(relative_path: str) -> Path:
    path = COSMOS3_AUDIOVISUAL_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    return path.resolve()


def compact_json_file(path: Path) -> str:
    return json.dumps(json.loads(path.read_text()), ensure_ascii=True, separators=(",", ":"))


def normalize_negative_prompt(value) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        parts = []
        for v in value.values():
            if isinstance(v, str):
                parts.append(v)
            elif isinstance(v, list):
                parts.extend(str(x) for x in v)
        return "\n".join(parts)
    return str(value)


def payload_dimensions(payload: dict) -> tuple[int, int]:
    if payload.get("resolution") == "720" and payload.get("aspect_ratio") == "16,9":
        return 720, 1280
    if payload.get("resolution") == "256" and payload.get("aspect_ratio") == "16,9":
        return 192, 320
    raise ValueError(f"Unsupported payload resolution/aspect ratio: {payload.get('resolution')} {payload.get('aspect_ratio')}")


def resolve_payload_path(payload_path: Path, value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return (payload_path.parent / path).resolve()


def create_payload(use_case: str, *, backend: str) -> tuple[Path, Path, str]:
    spec = ASSET_SETS[use_case]
    payload_dir = Path(os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"]) / backend / "payloads" / use_case
    output_dir = Path(os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"]) / backend / use_case
    payload_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    prompt_path = asset_path(spec["prompt"])
    negative_prompt = ""
    if spec["mode"] != "text2image":
        negative_prompt_path = asset_path(f"assets/negative_prompts/{spec['mode']}/neg_prompt.json") if not spec.get("negative_prompt") else spec["negative_prompt"]
        negative_prompt = normalize_negative_prompt(negative_prompt_path)
    payload_path = payload_dir / f"{use_case}.json"
    payload = {
        "model_mode": spec["mode"],
        "name": use_case,
        "prompt": compact_json_file(prompt_path),
        "negative_prompt": negative_prompt,
        "enable_sound": spec["enable_sound"],
        **FIXED_SAMPLING,
    }
    if spec["mode"] in {"image2video"}:
        image_path = asset_path(spec["image"])
        payload["vision_path"] = os.path.relpath(image_path, payload_path.parent)
    
    if spec["mode"] in {"video2video"}:
        video_path = asset_path(spec["video"])
        payload["vision_path"] = os.path.relpath(video_path, payload_path.parent)

    payload_path.write_text(json.dumps(payload, indent=2) + "\n")

    os.environ[f"COSMOS3_{backend.upper()}_{use_case.upper()}_INPUT"] = str(payload_path)
    os.environ[f"COSMOS3_{backend.upper()}_{use_case.upper()}_OUTPUT"] = str(output_dir)

    print(f"model:   {spec['model']}")
    print(f"payload: {payload_path}")
    print(f"output:  {output_dir}")
    print(f"prompt:  {prompt_path.relative_to(COSMOS_ROOT)}")
    if "vision_path" in payload:
        if payload["model_mode"] == "image2video":
            image_display_path = resolve_payload_path(payload_path, payload["vision_path"])
            print(f"image:   {image_display_path.relative_to(COSMOS_ROOT)}")
            display(Image(filename=str(image_display_path), width=420))
        elif payload["model_mode"] == "video2video":
            video_display_path = resolve_payload_path(payload_path, payload["vision_path"])
            print(f"video:   {video_display_path.relative_to(COSMOS_ROOT)}")
            display(Video(filename=str(video_display_path), width=420))
    print(json.dumps({k: payload[k] for k in ["model_mode", "name", "enable_sound", "num_steps", "guidance", "shift", "fps", "num_frames", "resolution", "aspect_ratio", "seed"]}, indent=2))
    return payload_path, output_dir, spec["model"]


import base64
import html
import json
import os
import subprocess
import time
from pathlib import Path
from IPython.display import HTML, display


def api_root_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if not normalized.endswith("/v1"):
        normalized = f"{normalized}/v1"
    return normalized


def video_api_url(base_url: str) -> str:
    return f"{api_root_url(base_url)}/videos"


def image_api_url(base_url: str) -> str:
    return f"{api_root_url(base_url)}/images/generations"

def build_sglang_video_form(payload: dict) -> dict[str, str]:
    height, width = payload_dimensions(payload)
    extra_params = {
        "use_resolution_template": False,
        "use_duration_template": False,
        "guardrails": True,
    }
    form = {
        "prompt": payload["prompt"],
        "negative_prompt": payload["negative_prompt"],
        "size": f"{width}x{height}",
        "num_frames": str(payload["num_frames"]),
        "fps": str(payload["fps"]),
        "num_inference_steps": str(payload["num_steps"]),
        "guidance_scale": str(payload["guidance"]),
        "flow_shift": str(payload["shift"]),
        "seed": str(payload["seed"]),
        "extra_params": json.dumps(extra_params, separators=(",", ":")),
    }
    if payload["enable_sound"]:
        form["generate_sound"] = "true"
        form["sound_duration"] = f"{payload['num_frames'] / payload['fps']:.3f}"
    return form


def build_sglang_image_body(payload: dict) -> dict:
    height, width = payload_dimensions(payload)
    return {
        "prompt": payload["prompt"],
        "negative_prompt": payload.get("negative_prompt", ""),
        "size": f"{width}x{height}",
        "n": 1,
        "num_inference_steps": payload["num_steps"],
        "guidance_scale": payload["guidance"],
        "flow_shift": payload["shift"],
        "seed": payload["seed"],
        "response_format": "b64_json",
        "extra_params": {
            "use_resolution_template": False,
            "guardrails": True,
        },
    }


def post_video(*, payload_path: Path, payload: dict, output_path: Path, model: str) -> None:
    url = video_api_url(SGLANG_ENDPOINTS[model])
    api_key = os.environ.get("COSMOS3_SGLANG_API_KEY") or None
    tmp_path = Path(f"{output_path}.tmp")
    error_path = Path(f"{output_path}.error.txt")
    if tmp_path.exists():
        tmp_path.unlink()
    if error_path.exists():
        error_path.unlink()

    cmd = [
        "curl",
        "-sS",
        "--fail-with-body",
        "-X",
        "POST",
        url,
        "-H",
        "Accept: video/mp4",
    ]
    if api_key is not None:
        cmd += ["-H", f"Authorization: Bearer {api_key}"]

    for key, value in build_sglang_video_form(payload).items():
        cmd += ["--form-string", f"{key}={value}"]

    if payload["model_mode"] == "image2video":
        image_path = resolve_payload_path(payload_path, payload["vision_path"])
        cmd += ["-F", f"input_reference=@{image_path}"]

    if payload["model_mode"] == "video2video":
        video_path = resolve_payload_path(payload_path, payload["vision_path"])
        cmd += ["-F", f"video_reference=@{video_path};type=video/mp4"]

    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode != 0:
        error_path.write_text((result.stdout or "") + (result.stderr or ""))
        raise RuntimeError(f"SGLang request failed with exit code {result.returncode}; see {error_path}")

    # poll output video
    initial = json.loads(result.stdout)
    video_id = initial["id"]

    while True:
        poll = subprocess.run(
            ["curl", "-sS", "--fail-with-body", f"{url}/{video_id}"],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )
        if poll.returncode != 0:
            error_path.write_text((poll.stdout or "") + (poll.stderr or ""))
            raise RuntimeError(f"SGLang video poll failed; see {error_path}")

        final = json.loads(poll.stdout)
        if final.get("status") == "completed":
            break
        if final.get("status") in {"failed", "cancelled"}:
            error_path.write_text(json.dumps(final, indent=2))
            raise RuntimeError(f"SGLang video failed; see {error_path}")
        time.sleep(5)

    download = subprocess.run(
        ["curl", "-sS", "--fail-with-body", f"{url}/{video_id}/content", "-o", str(tmp_path)],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if download.returncode != 0:
        error_path.write_text(json.dumps(final, indent=2) + "\n" + (download.stderr or ""))
        raise RuntimeError(f"SGLang video download failed; see {error_path}")

    tmp_path.replace(output_path)


def post_image(*, payload: dict, output_path: Path, model: str) -> None:
    url = image_api_url(SGLANG_ENDPOINTS[model])
    api_key = os.environ.get("COSMOS3_SGLANG_API_KEY") or None
    tmp_path = Path(f"{output_path}.tmp")
    error_path = Path(f"{output_path}.error.txt")
    if tmp_path.exists():
        tmp_path.unlink()
    if error_path.exists():
        error_path.unlink()

    cmd = [
        "curl",
        "-sS",
        "--fail-with-body",
        "-X",
        "POST",
        url,
        "-H",
        "Content-Type: application/json",
    ]
    if api_key is not None:
        cmd += ["-H", f"Authorization: Bearer {api_key}"]
    cmd += ["-d", json.dumps(build_sglang_image_body(payload), separators=(",", ":"))]

    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode != 0:
        error_path.write_text((result.stdout or "") + (result.stderr or ""))
        raise RuntimeError(f"SGLang image request failed with exit code {result.returncode}; see {error_path}")
    try:
        response = json.loads(result.stdout)
        b64_json = response["data"][0]["b64_json"]
        tmp_path.write_bytes(base64.b64decode(b64_json))
    except Exception as exc:
        error_path.write_text((result.stdout or "") + (result.stderr or ""))
        raise RuntimeError(f"Could not decode SGLang image response; see {error_path}") from exc
    tmp_path.replace(output_path)


def run_sglang_payload(payload_path: Path, output_dir: str | Path, *, model: str) -> Path:
    payload_path = Path(payload_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    payload = json.loads(payload_path.read_text())
    output_ext = ".png" if payload["model_mode"] == "text2image" else ".mp4"
    output_path = output_dir / f"{payload['name']}{output_ext}"
    endpoint = image_api_url(SGLANG_ENDPOINTS[model]) if payload["model_mode"] == "text2image" else video_api_url(SGLANG_ENDPOINTS[model])
    print("endpoint:", endpoint)
    print("payload:", payload_path)
    print("output:", output_path)
    if payload["model_mode"] == "image2video":
        print("input image:", resolve_payload_path(payload_path, payload["vision_path"]))
    t0 = time.time()
    if payload["model_mode"] == "text2image":
        post_image(payload=payload, output_path=output_path, model=model)
    else:
        post_video(payload_path=payload_path, payload=payload, output_path=output_path, model=model)
    print(f"wrote {output_path} in {time.time() - t0:.1f}s")
    return output_path


def display_video(path: Path, *, width: int = 720) -> None:
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    label = html.escape(str(path))
    markup = f"""
<video controls playsinline preload="metadata" width="{width}" style="max-width: 100%; background: #000;">
  <source src="data:video/mp4;base64,{data}" type="video/mp4">
</video>
<div style="font-family: monospace; font-size: 12px; margin-top: 4px;">{label}</div>
"""
    display(HTML(markup))


def view_run(output_dir: str | Path) -> None:
    output_dir = Path(output_dir)
    videos = [
        path
        for path in sorted(output_dir.rglob("*.mp4"))
        if not path.name.endswith(("_preview.mp4", "_browser.mp4"))
    ]
    images = sorted(output_dir.rglob("*.png"))
    if not videos and not images:
        print(f"No generated media found under {output_dir}")
        return
    for src in videos:
        print(f"source: {src} ({src.stat().st_size // 1024} KB)")
        display_video(src)
    for src in images:
        print(f"source: {src} ({src.stat().st_size // 1024} KB)")
        display(Image(filename=str(src), width=720))


Run each use case top-to-bottom: create the JSON payload, run inference, then view the generated media. The examples are grouped by model size below. The Cosmos3-Nano and Cosmos3-Super sections are independent, so you can run just one.


# Cosmos3-Nano Examples

Use cases for the `Cosmos3-Nano` model. This section is self-contained; you can run it without the Cosmos3-Super section below.


## Nano: Text to Image

Nano text-to-image generation using a structured JSON prompt.

### Create Payload


In [ ]:
t2i_payload, t2i_output, t2i_model = create_payload("t2i", backend="sglang")


### Run


In [ ]:
run_sglang_payload(t2i_payload, t2i_output, model="Cosmos3-Nano")


### View Results


In [ ]:
view_run(t2i_output)


## Nano: Text to Video Without Audio

Nano text-to-video generation with audio disabled.

### Create Payload


In [ ]:
t2v_nano_noaudio_payload, t2v_nano_noaudio_output, t2v_nano_noaudio_model = create_payload("t2v_nano_noaudio", backend="sglang")


### Run


In [ ]:
run_sglang_payload(t2v_nano_noaudio_payload, t2v_nano_noaudio_output, model="Cosmos3-Nano")


### View Results


In [ ]:
view_run(t2v_nano_noaudio_output)


## Nano: Text to Video with Audio

Nano text-to-video generation with generated audio.

### Create Payload


In [ ]:
t2vs_payload, t2vs_output, t2vs_model = create_payload("t2vs", backend="sglang")


### Run


In [ ]:
run_sglang_payload(t2vs_payload, t2vs_output, model="Cosmos3-Nano")


### View Results


In [ ]:
view_run(t2vs_output)


## Nano: Image to Video Without Audio

Nano image-to-video generation using its paired image asset, with audio disabled.

### Create Payload


In [ ]:
i2v_nano_noaudio_payload, i2v_nano_noaudio_output, i2v_nano_noaudio_model = create_payload("i2v_nano_noaudio", backend="sglang")


### Run


In [ ]:
run_sglang_payload(i2v_nano_noaudio_payload, i2v_nano_noaudio_output, model="Cosmos3-Nano")


### View Results


In [ ]:
view_run(i2v_nano_noaudio_output)


## Nano: Image to Video with Audio

Nano image-to-video generation using its paired image asset and generated audio.

### Create Payload


In [ ]:
i2vs_payload, i2vs_output, i2vs_model = create_payload("i2vs", backend="sglang")


### Run


In [ ]:
run_sglang_payload(i2vs_payload, i2vs_output, model="Cosmos3-Nano")


### View Results


In [ ]:
view_run(i2vs_output)


## Nano: Video to Video with Audio

Nano video-to-video generation with sound.

### Create Payload


In [ ]:
v2vs_payload, v2vs_output, v2vs_model = create_payload("v2vs", backend="sglang")

### Run


In [ ]:
run_sglang_payload(v2vs_payload, v2vs_output, model="Cosmos3-Nano")


### View Results


In [ ]:
view_run(v2vs_output)


# Cosmos3-Super Examples

The same use cases for the larger `Cosmos3-Super` model. This section is self-contained; you can run it without the Cosmos3-Nano section above.


## Super: Text to Image

Super text-to-image generation using the same structured JSON prompt.

### Create Payload


In [ ]:
t2i_super_payload, t2i_super_output, t2i_super_model = create_payload("t2i_super", backend="sglang")


### Run


In [ ]:
run_sglang_payload(t2i_super_payload, t2i_super_output, model="Cosmos3-Super")


### View Results


In [ ]:
view_run(t2i_super_output)


## Super: Text to Video Without Audio

Super text-to-video generation with audio disabled.

### Create Payload


In [ ]:
t2v_super_noaudio_payload, t2v_super_noaudio_output, t2v_super_noaudio_model = create_payload("t2v_super_noaudio", backend="sglang")


### Run


In [ ]:
run_sglang_payload(t2v_super_noaudio_payload, t2v_super_noaudio_output, model="Cosmos3-Super")


### View Results


In [ ]:
view_run(t2v_super_noaudio_output)


## Super: Image to Video Without Audio

Super image-to-video generation using its paired image asset, with audio disabled.

### Create Payload


In [ ]:
i2v_super_noaudio_payload, i2v_super_noaudio_output, i2v_super_noaudio_model = create_payload("i2v_super_noaudio", backend="sglang")


### Run


In [ ]:
run_sglang_payload(i2v_super_noaudio_payload, i2v_super_noaudio_output, model="Cosmos3-Super")


### View Results


In [ ]:
view_run(i2v_super_noaudio_output)
